# Milestone 1: NLP Foundation & Semantic Similarity

**Deadline:**  `Jun 24, 2026`

- Perform Text cleaning, tokenization, and handling missing data.
- Generate embeddings using baseline models like TF-IDF and Word2Vec.
- Compute cosine similarity between a 'prompt' and 'options' and understand its concepts.
- Calculate mAP@3 and understand the concept of mAP.

## Import libraries

In [1]:
import string
from pathlib import Path
from typing import List, LiteralString

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Config

In [2]:
DATA_DIR = Path("../dataset/")

## Load data

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

## Question 1

Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the *most frequent* option and the least frequent option?

In [4]:
occurs = train["answer"].value_counts()

occurs.min() + occurs.max()

np.int64(814)

## Question 2

After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's `string.punctuation`), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of `train.csv`?

In [5]:
lower_prompt = train["prompt"].str.lower()

punct_table = str.maketrans("", "", string.punctuation)
cleaned_prompt = lower_prompt.str.translate(punct_table)

vocabulary = set()
for p in cleaned_prompt:
    words = p.split()
    vocabulary.update(words)

len(vocabulary)

859

## Question 3

Using the cleaned prompt from Row ID 1, filter out the standard English stop words using `sklearn.feature_extraction.text.ENGLISH_STOP_WORDS`. How many words are left in the prompt for Row ID 1 after filtering?

In [6]:
len([w for w in cleaned_prompt[0].split() if w not in ENGLISH_STOP_WORDS])

13

## Question 4

Fit a default `TfidfVectorizer(stop_words='english')` on a list containing all the combined text of the prompts and options in `train.csv`. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?

In [7]:
comb_prompts = train[["prompt", "A", "B", "C", "D", "E"]].astype(str).values.flatten()

In [8]:
vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(comb_prompts)

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",'english'
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.

In [9]:
len(vectorizer.vocabulary_)

2762

## Question 5

Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).

In [10]:
p_vec = vectorizer.transform([train.iloc[0]["prompt"]])
op_vec = vectorizer.transform([train.iloc[0]["A"]])

round(cosine_similarity(p_vec, op_vec)[0][0], 4)

np.float64(0.2328)

## Question 6

Expand the logic from Question 4: For every row in `train.csv`, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.

In [11]:
p_vec = vectorizer.transform(train["prompt"])
sim_score = {}

for c in ["A", "B", "C", "D", "E"]:
    op_vec = vectorizer.transform(train[c])

    dot_prod = np.array(p_vec.multiply(op_vec).sum(axis=1)).flatten()  # type: ignore[missing-attribute] # reason: p_vec is a sparse matrix and does have the attribute 'multiply'
    sim_score[c] = dot_prod

In [12]:
sim_df = pd.DataFrame(sim_score)

pred = sim_df.idxmax(axis=1)
(pred == train["answer"]).mean() * 100

np.float64(13.700000000000001)

## Question 7

If the ground truth answer for a question is `C`, what is the **MAP@3** score if a model predicts `C A B`?

In [13]:
def map_3(truth: LiteralString, pred: List[LiteralString]) -> float:
    if truth not in pred:
        return 0.0

    return 1.0 / (pred.index(truth) + 1)

In [14]:
map_3("C", "C A B".split())

1.0

## Question 8

If the ground truth answer for a question is `B`, what is the **MAP@3** score if a model predicts `D B E`?

In [15]:
map_3("B", "D B E".split())

0.5

## Question 9

**The Majority Class Baseline:** Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall **MAP@3** score of this "Majority Class" baseline on `train.csv`?

In [16]:
guess = occurs.index.tolist()[:3]
np.mean([map_3(r["answer"], guess) for _, r in train.iterrows()])

np.float64(0.42125)

## Question 10

**The TF-IDF Pipeline:** Build a basic pipeline that evaluates every row in `train.csv`. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average **MAP@3** score of this TF-IDF pipeline across the entire training set?

In [17]:
options = np.array(["A", "B", "C", "D", "E"])
pred = options[np.argsort(sim_df.values, axis=1)[:, ::-1]][:, :3]

np.mean(
    [map_3(r["answer"], list(guess)) for guess, (_, r) in zip(pred, train.iterrows())]
)

np.float64(0.27616666666666667)